# ICA: the Bell-Sejnowski (infomax) algorithm

_Machine Learning — more_

**Learn the unmixing matrix W by gradient ascent on the log-likelihood of the data.**

The ml-ica lesson sets up the problem: recordings $x = As$ are a blend of
       independent sources $s$, and to unmix them we need the unmixing matrix
       $W = A^{-1}$ so that $\hat{s} = Wx$. But it stops at "ICA finds the $W$ that makes the
       outputs independent" without saying how. This lesson is the how.

       The Bell-Sejnowski algorithm (also called infomax) turns the search for $W$
       into ordinary maximum likelihood: pick the $W$ that makes the observed recordings most
       probable. We write down the probability of the data as a function of $W$, take the log, and
       climb it with gradient ascent. Each step nudges $W$ uphill until the recovered signals
       look like independent, non-Gaussian sources.

---

This notebook is a practice scaffold for the **ICA: the Bell-Sejnowski (infomax) algorithm** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — NumPy

In [ ]:
import numpy as np
np.random.seed(0)

# --- 1. two independent NON-Gaussian sources (Laplace: peaky, heavy-tailed) ---
m = 4000
S = np.vstack([np.random.laplace(0, 1, m),
               np.random.laplace(0, 1, m)])          # 2 x m true sources
S = S / S.std(axis=1, keepdims=True)                 # unit scale

# --- 2. mix with a KNOWN A, then center (algorithm assumes zero mean) ---
A = np.array([[1.0, 0.7],
              [0.4, 1.0]])
X = A @ S
X = X - X.mean(axis=1, keepdims=True)                # 2 x m observed mixtures

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# --- 3. Bell-Sejnowski stochastic gradient ASCENT on the log-likelihood ---
W = np.eye(2)                                        # start from the identity
for epoch in range(40):
    alpha = 0.01 / (1.0 + 0.05 * epoch)              # gently anneal the step size
    for i in np.random.permutation(m):
        x = X[:, i:i+1]                              # one recording, 2 x 1
        g = sigmoid(W @ x)                           # 2 x 1
        grad = (1 - 2 * g) @ x.T + np.linalg.inv(W.T)   # the update's gradient
        W = W + alpha * grad                         # ASCENT: add the gradient

# --- 4. did we recover A^-1 (up to scale / permutation)? ---
print("recovered W =\n", np.round(W, 3))
print("W @ A (should be a scaled permutation) =\n", np.round(W @ A, 3))

R = W @ X                                            # recovered sources
def abscorr(a, b):
    a, b = a - a.mean(), b - b.mean()
    return abs((a * b).sum() / np.sqrt((a*a).sum() * (b*b).sum()))

print("\n|corr| of each recovered source with the true sources:")
for r in range(2):
    cs = [abscorr(R[r], S[j]) for j in range(2)]
    print(f"  recovered {r}: max |corr| = {max(cs):.3f} (-> true source {int(np.argmax(cs))})")
# recovered W =
#  [[ 2.649 -1.869]
#   [-1.058  2.728]]
# W @ A (should be a scaled permutation) =
#  [[ 1.901 -0.015]
#   [ 0.033  1.988]]      # off-diagonals ~0  ->  W = A^-1 up to scale/order
# |corr| of each recovered source with the true sources:
#   recovered 0: max |corr| = 1.000 (-> true source 0)
#   recovered 1: max |corr| = 1.000 (-> true source 1)

## Visualize the data & results

_After running Bell-Sejnowski ICA, how well does each recovered source match its true source, compared with the raw mixed signals it started from?_

In [ ]:
import numpy as np
np.random.seed(0)

m = 4000
S = np.vstack([np.random.laplace(0,1,m), np.random.laplace(0,1,m)])
S = S / S.std(axis=1, keepdims=True)
A = np.array([[1.0,0.7],[0.4,1.0]])
X = A @ S; X = X - X.mean(axis=1, keepdims=True)

sig = lambda z: 1.0/(1.0+np.exp(-z))
W = np.eye(2)
for epoch in range(40):
    alpha = 0.01/(1.0 + 0.05*epoch)
    for i in np.random.permutation(m):
        x = X[:, i:i+1]; g = sig(W @ x)
        W = W + alpha*((1-2*g) @ x.T + np.linalg.inv(W.T))

R = W @ X
def c(a, b):
    a, b = a-a.mean(), b-b.mean()
    return abs((a*b).sum()/np.sqrt((a*a).sum()*(b*b).sum()))

rec = [max(c(R[r], S[0]), c(R[r], S[1])) for r in range(2)]   # after ICA
mix = [max(c(X[r], S[0]), c(X[r], S[1])) for r in range(2)]   # before (raw mics)
print("recovered |corr|:", [round(v,3) for v in rec])  # -> [1.0, 1.0]
print("mixed     |corr|:", [round(v,3) for v in mix])  # -> [0.818, 0.928]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Why does the term $\log\lvert W\rvert$ appear in the log-likelihood $\ell(W)$? What would go wrong if you dropped it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall we model the sources $s = Wx$ with a density $p_s$, but we observe $x$, not $s$. — _The likelihood must be the probability of the observed $x$, so we need the density of $x$, not of $s$._
- Apply the change-of-variables rule: $p_x(x) = p_s(Wx)\lvert\det W\rvert$. — _A linear map $W$ rescales volumes by $\lvert\det W\rvert$, so density must rescale by the same factor to stay a valid probability density (see la-determinant)._
- Take the log: the determinant factor becomes the additive $\log\lvert W\rvert$. — _Logs turn the product into a sum, giving the term in $\ell(W)$._

**Answer:** It is the change-of-variables Jacobian. We model the sources but observe the mixtures, so the density of $x$ picks up a factor $\lvert\det W\rvert$ that accounts for how $W$ rescales volume; its log is $\log\lvert W\rvert$. Drop it and the likelihood no longer penalizes $W$ shrinking toward singular &mdash; gradient ascent would happily drive $\det W \to 0$, collapsing all recovered sources onto one direction instead of unmixing them.

</details>

**Problem 2.** In the update, what is the role of the $(W^{\top})^{-1}$ term, and what does the natural-gradient variant do with it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify $(W^{\top})^{-1}$ as the gradient of the $\log\lvert W\rvert$ term. — _The matrix-calculus identity $\nabla_W \log\lvert W\rvert = (W^{\top})^{-1}$._
- See that as $\det W \to 0$ this term blows up. — _It pushes $W$ away from singular, keeping it invertible &mdash; a real unmixing matrix._
- Recall the natural gradient multiplies the whole gradient by $W^{\top}W$. — _That cancels the matrix inverse, giving the cheaper, better-conditioned update $(I + (1-2g)(Wx)^{\top})W$._

**Answer:** $(W^{\top})^{-1}$ is the gradient of $\log\lvert W\rvert$; it keeps $W$ invertible by pushing it away from singular matrices (where the likelihood is $-\infty$). The natural-gradient variant multiplies the gradient by $W^{\top}W$, which cancels the inverse and replaces it with the cheaper, more stable update used in real infomax ICA.

</details>

**Problem 3.** You run the update and $W$ converges, but $WA$ comes out as $\begin{bmatrix}0 & 1.9\\ 2.0 & 0\end{bmatrix}$ instead of the identity. Did ICA fail?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check whether $WA$ is a scaled permutation matrix (one nonzero per row/column). — _If so, the recovered sources are the true sources reordered and rescaled._
- Recognize that ICA is only identifiable up to scale, sign, and permutation. — _Independence is preserved under reordering and rescaling, so the likelihood cannot prefer one labeling/scale over another._
- Match each recovered source to a true one (by absolute correlation) before judging. — _Anchoring resolves the permutation/sign so you can compare._

**Answer:** No, it succeeded. $WA$ is a scaled permutation (zeros on the diagonal, nonzeros off it), meaning recovered source 1 is true source 2 rescaled, and vice versa. ICA recovers sources only up to scale, sign, and order, so a clean permutation is a perfect result &mdash; just re-pair and rescale the components.

</details>